# Experiment 7.0 — Hierarchical context SNN

Analysis-only notebook. It consumes finalized method-level aggregate artifacts and intentionally does **not** display individual training runs, per-seed learning curves, or per-run rasters.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path('artifacts/experiment_7_0_hierarchical_context_snn/hierarchical_context_v1')
if not ROOT.exists():
    ROOT = Path('notebooks') / ROOT
summary = pd.read_csv(ROOT / 'summary.csv')
paired = pd.read_csv(ROOT / 'paired_neuron_deltas.csv')
method_deltas = pd.read_csv(ROOT / 'method_deltas.csv')
firing = pd.read_csv(ROOT / 'firing_summary.csv')


## Method-level test performance

In [ ]:
display_cols = [
    'method', 'synapse_mode', 'n_runs',
    'val_ba_mean', 'val_ba_std',
    'test_ba_mean', 'test_ba_std',
    'test_accuracy_mean', 'test_macro_f1_mean',
]
summary[display_cols].sort_values('test_ba_mean', ascending=False).reset_index(drop=True)

In [ ]:
frame = summary.copy()
labels = frame['method'] + '\n' + frame['synapse_mode']
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.bar(range(len(frame)), frame['test_ba_mean'], yerr=frame['test_ba_std'].fillna(0), capsize=3)
ax.set_xticks(range(len(frame)))
ax.set_xticklabels(labels, rotation=35, ha='right')
ax.set_ylim(0, 1)
ax.set_ylabel('Test balanced accuracy')
ax.set_title('Exp7.0 method comparison — mean ± SD')
plt.tight_layout()
plt.show()

## Paired neuron-dynamics effect

Positive values mean normalized/unit-DC synaptic dynamics outperform the matched legacy run.

In [ ]:
paired_summary = (
    paired.groupby('method')['normalized_minus_legacy_test_ba']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
paired_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.bar(range(len(paired_summary)), paired_summary['mean'], yerr=paired_summary['std'].fillna(0), capsize=3)
ax.axhline(0, linewidth=1)
ax.set_xticks(range(len(paired_summary)))
ax.set_xticklabels(paired_summary['method'], rotation=25, ha='right')
ax.set_ylabel('Normalized - legacy test BA')
ax.set_title('Paired neuron-dynamics effect')
plt.tight_layout()
plt.show()

## Architecture contrasts

These contrasts directly test whether multiscale/additive/gated context improves over WHAT-only, and whether context gain improves over additive context.

In [ ]:
contrast_summary = (
    method_deltas.groupby(['synapse_mode', 'contrast'])['test_ba_delta']
    .agg(['mean', 'std', 'count'])
    .reset_index()
)
contrast_summary

## Aggregated firing and zero-tail persistence

This table is method-level aggregation only. It is intended to diagnose whether long legacy dynamics produce persistent firing after the valid endpoint.

In [ ]:
firing.sort_values(['method', 'synapse_mode', 'layer']).reset_index(drop=True)

In [ ]:
hier_firing = firing[firing['synapse_mode'].isin(['legacy', 'normalized'])].copy()
hier_firing['label'] = hier_firing['method'] + ' / ' + hier_firing['synapse_mode'] + ' / ' + hier_firing['layer']
fig, ax = plt.subplots(figsize=(14, 5.5))
ax.bar(range(len(hier_firing)), hier_firing['zero_tail_firing_fraction_mean'])
ax.set_xticks(range(len(hier_firing)))
ax.set_xticklabels(hier_firing['label'], rotation=55, ha='right')
ax.set_ylabel('Mean zero-tail firing fraction')
ax.set_title('Persistent firing after valid endpoint')
plt.tight_layout()
plt.show()